# Anotador TDAH · 03/04 · Backend `tool_calling`

Define la anotación como una función (`registrar_anotacion_clinica`) con su esquema de argumentos y llama a `/api/chat` de Ollama con `tools`. El modelo no escribe JSON en el texto: **invoca la función** y devuelve los argumentos ya estructurados en `message.tool_calls`.

He añadido una instrucción al prompt para forzar la tool call (si no, el modelo tiende a escribir el JSON en el cuerpo del mensaje), y se envía `think: False` como los modelos de razonamiento (Qwen3) no gasten los tokens pensando. Si el modelo no emite la tool call, hay fallback a extraer el JSON del texto.

## 1 · Parámetros

In [1]:
import datetime as dt
import json
import sqlite3
import time

import pandas as pd

SEMANA       = 1            # semana de seguimiento (el dataset llega a la 24)
PACIENTES    = None         # None = todos los de la semana; o lista: ["P001", "P003"]
REPETICIONES = 3            # veces que se anota cada entrada
TEMPERATURA  = 0.7
MODELO       = "gemma4:e4b" 

# --- Rutas y conexión ---
RUTA_BD     = "datos/anotador.db"
OLLAMA_URL  = "http://127.0.0.1:11002"   
INSTRUMENTO = "instrumentos/brief2.json"

BACKEND     = "tool_calling"
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.date.today():%Y%m%d}"
print(f"Código de experimento: {EXPERIMENTO}")

Código de experimento: tool_calling-s1-t0.7-20260713


## 2 · Datos

Las entradas (texto libre de los padres) de la semana elegida, con el contexto del paciente (edad calculada a la fecha de la observación, sexo, quién informa).

In [2]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

entradas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    JOIN referencia_sintetica r USING (id_entrada)
    WHERE r.semana = ?
    ORDER BY e.id_paciente
    ''',
    con, params=[SEMANA],
)
if PACIENTES:
    entradas = entradas[entradas["id_paciente"].isin(PACIENTES)]


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


entradas["edad"] = [
    calcular_edad(n, f) for n, f in zip(entradas["fecha_nacimiento"], entradas["fecha"])
]

print(f"Semana {SEMANA}: {len(entradas)} entradas de {entradas['id_paciente'].nunique()} pacientes")
entradas[["id_entrada", "id_paciente", "informante", "edad", "sexo", "texto"]].head()

Instrumento: BRIEF-2 Familia (63 ítems)
Semana 1: 30 entradas de 30 pacientes


,id_entrada,id_paciente,informante,edad,sexo,texto
0,1,PAC001,madre,7,masculino,"Hoy ha sido un día horrible, la verdad. Marco ..."
1,25,PAC002,madre,11,femenino,"Hola, soy la madre de Lucía. Nos dijeron que t..."
2,49,PAC003,padre,15,masculino,Soy el padre de Alejandro. La psiquiatra nos h...
3,73,PAC004,madre,6,femenino,"Somos los padres de Sofía, tiene 6 años. Esta ..."
4,97,PAC005,madre,8,masculino,Soy la madre de Diego. Diego vive conmigo de l...


## 3 · Prompts

El prompt de sistema se construye desde el instrumento (`brief2.json`): catálogo de ítems, escalas y niveles de alerta. El de usuario lleva el contexto del paciente y su texto.

In [ ]:
COMILLAS = '"' * 3  


def construir_prompt_sistema(instrumento):
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}" for it in instrumento["items"]
    )
    escalas = "\n".join(f"  - {e}: {d}" for e, d in instrumento["escalas"].items())
    n = instrumento["niveles_alerta"]
    return f'''Eres un {instrumento["rol_anotador"]}.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
{instrumento["nombre"]}.

## CATÁLOGO DE ÍTEMS ({len(instrumento["items"])} ítems)
{catalogo}

## ESCALAS
{escalas}

## NIVELES DE ALERTA
{" | ".join(n)}

## INSTRUCCIONES DE SALIDA
Responde ÚNICAMENTE con un objeto JSON válido, sin texto antes ni después, sin markdown.
Estructura requerida:
{{
  "items_detectados": [lista de números de ítem observables en el texto],
  "escalas_afectadas": [lista de escalas correspondientes],
  "nivel_alerta": "{n[0]}|{n[1]}|{n[2]}",
  "nota_clinica": "resumen clínico de 1-3 frases para el médico",
  "justificacion": "explicación del razonamiento (para auditoría)"
}}'''


def construir_prompt_usuario(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años
- Sexo: {e.sexo}
- Informante: {e.informante}

## TEXTO DEL PADRE/MADRE
{COMILLAS}{e.texto}{COMILLAS}

Analiza el texto y genera el JSON de anotación clínica.'''


prompt_sistema = construir_prompt_sistema(instrumento)
print(prompt_sistema[:400] + "\n[...]")

Eres un psicólogo clínico infantil especializado en TDAH y en el instrumento BRIEF-2.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
BRIEF-2 Familia.

## CATÁLOGO DE ÍTEMS (63 ítems)
  1: [inhibicion] Es inquieto o inquieta.
  2: [flexibilidad] Se resiste o le cuesta acept
[...]


## 4 · El backend


In [4]:
import re

import requests


def extraer_json(texto):
    '''Intenta sacar el primer JSON válido del texto que devuelve el modelo.'''
    if not texto:
        return None
    try:
        return json.loads(texto.strip())
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{.*\}", texto, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    limpio = re.sub(r"```(?:json)?", "", texto).strip()
    try:
        return json.loads(limpio)
    except json.JSONDecodeError:
        return None

In [5]:
from pydantic import BaseModel, Field


class Anotacion(BaseModel):
    items_detectados: list[int] = Field(default_factory=list)
    escalas_afectadas: list[str] = Field(default_factory=list)
    nivel_alerta: str = "bajo"
    nota_clinica: str = ""
    justificacion: str = ""

In [6]:
DIRECTIVA_TOOL = (
    "\n\nIMPORTANTE: para entregar el resultado DEBES invocar la función "
    "`registrar_anotacion_clinica` con los argumentos correspondientes. "
    "No respondas con texto ni con JSON en el cuerpo del mensaje."
)

TOOL = {
    "type": "function",
    "function": {
        "name": "registrar_anotacion_clinica",
        "description": "Registra la anotación clínica estructurada de la observación.",
        "parameters": Anotacion.model_json_schema(),
    },
}


def anotar(prompt_sistema, prompt_usuario):
    '''Llama al modelo y devuelve (anotacion | None, respuesta_cruda).'''
    payload = {
        "model": MODELO,
        "messages": [
            {"role": "system", "content": prompt_sistema + DIRECTIVA_TOOL},
            {"role": "user", "content": prompt_usuario},
        ],
        "tools": [TOOL],
        "stream": False,
        "think": False,
        "options": {"temperature": TEMPERATURA, "num_predict": 2048},
    }
    try:
        r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=180)
        r.raise_for_status()
    except requests.HTTPError:
        payload.pop("think", None)  # algunos modelos no aceptan `think`
        r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=180)
        r.raise_for_status()
    msg = r.json().get("message", {})
    tool_calls = msg.get("tool_calls") or []
    if tool_calls:
        args = tool_calls[0]["function"]["arguments"]
        if isinstance(args, str):
            args = json.loads(args)
        return args, json.dumps(args, ensure_ascii=False)
    contenido = msg.get("content", "")   # el modelo no llamó a la función
    return extraer_json(contenido), contenido

## 5 · Una anotación de ejemplo

Antes de lanzar el experimento completo, una sola entrada para ver la anotación final que produce este backend.

In [7]:
ejemplo = entradas.iloc[0]
print(f"Paciente {ejemplo.id_paciente} · {ejemplo.informante} · semana {SEMANA}")
print(f"Texto: {ejemplo.texto[:200]}...\n")

t0 = time.time()
anotacion, cruda = anotar(prompt_sistema, construir_prompt_usuario(ejemplo))
print(f"Latencia: {time.time() - t0:.1f}s\n")

if anotacion is None:
    print("[FALLO DE FORMATO] El modelo no devolvió un JSON válido:")
    print(cruda[:500])
else:
    print("ANOTACIÓN FINAL:")
    print(json.dumps(anotacion, indent=2, ensure_ascii=False))

Paciente PAC001 · madre · semana 1
Texto: Hoy ha sido un día horrible, la verdad. Marco lleva tres semanas desde el diagnóstico y yo sigo sin saber muy bien cómo manejarlo. Esta mañana no había manera de que se sentara a desayunar, estaba sal...

Latencia: 10.5s

ANOTACIÓN FINAL:
{
  "escalas_afectadas": [
    "inhibicion",
    "supervision_conducta",
    "memoria_trabajo"
  ],
  "items_detectados": [
    1,
    4,
    3
  ],
  "justificacion": "Se detectaron tres ítems clave: el movimiento constante y saltar de la silla (Ítem 1: Inhibición), empujar a otro niño sin darse cuenta del impacto (Ítem 20: Supervisión Conducta) y dificultad para recordar una secuencia de tres tareas (Ítem 3: Memoria Trabajo). Estos apuntan directamente a problemas de control motor, conciencia social e intervención ejecutiva.",
  "nivel_alerta": "alto",
  "nota_clinica": "Se observa un patrón de hiperactividad motora significativa en el contexto doméstico y dificultades notorias en la secuenciación de tareas 

## 6. Experimento Toll Calling

### 6.1. Repetir experimento mismo día
Mismo día, mismo código (borrar y repetir)

In [ ]:
# Borra todas las filas de este experimento tool_calling y empieza de cero
con.execute("DELETE FROM experimento WHERE codigo = ?", [EXPERIMENTO])
con.commit()
print(f"Borradas filas de '{EXPERIMENTO}'. Listo para relanzar.")

### 6.2. Repetrir experimento
Mismo día, código nuevo (conservar el anterior)

In [ ]:
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.datetime.now():%Y%m%d-%H%M}"

### 6.3. · Experimento

Anota cada entrada de la semana `REPETICIONES` veces y guarda cada resultado en la tabla `experimento` con el código `EXPERIMENTO`.

In [8]:
con.execute('''
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,      -- código del experimento (para comparar)
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,               -- JSON: [int]
    escalas_afectadas TEXT,               -- JSON: [str]
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL
)''')
con.commit()

total = len(entradas) * REPETICIONES
print(f"Experimento '{EXPERIMENTO}': {len(entradas)} entradas × {REPETICIONES} repeticiones "
      f"= {total} llamadas al modelo")

hechas = 0
for _, e in entradas.iterrows():
    prompt_usuario = construir_prompt_usuario(e)
    for rep in range(REPETICIONES):
        t0 = time.time()
        anotacion, cruda = anotar(prompt_sistema, prompt_usuario)
        latencia = time.time() - t0
        ok = anotacion is not None
        a = anotacion or {}
        con.execute(
            "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
            "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
            "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (EXPERIMENTO, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
             MODELO, TEMPERATURA, SEMANA, e.id_paciente, int(e.id_entrada), rep,
             int(ok), json.dumps(a.get("items_detectados", [])),
             json.dumps(a.get("escalas_afectadas", [])), a.get("nivel_alerta"),
             a.get("nota_clinica"), a.get("justificacion"), latencia),
        )
        con.commit()
        hechas += 1
        estado = "ok" if ok else "FALLO DE FORMATO"
        print(f"  [{hechas:>3}/{total}] {e.id_paciente} rep {rep + 1} → {estado} ({latencia:.1f}s)")

print(f"\nGuardado en la tabla `experimento` con codigo = '{EXPERIMENTO}'")

Experimento 'tool_calling-s1-t0.7-20260713': 30 entradas × 3 repeticiones = 90 llamadas al modelo
  [  1/90] PAC001 rep 1 → ok (2.5s)
  [  2/90] PAC001 rep 2 → ok (2.5s)
  [  3/90] PAC001 rep 3 → ok (2.4s)
  [  4/90] PAC002 rep 1 → ok (2.4s)
  [  5/90] PAC002 rep 2 → ok (2.3s)
  [  6/90] PAC002 rep 3 → ok (2.4s)
  [  7/90] PAC003 rep 1 → ok (2.9s)
  [  8/90] PAC003 rep 2 → ok (2.6s)
  [  9/90] PAC003 rep 3 → ok (2.5s)
  [ 10/90] PAC004 rep 1 → ok (2.7s)
  [ 11/90] PAC004 rep 2 → ok (2.4s)
  [ 12/90] PAC004 rep 3 → ok (2.4s)
  [ 13/90] PAC005 rep 1 → ok (2.5s)
  [ 14/90] PAC005 rep 2 → ok (3.0s)
  [ 15/90] PAC005 rep 3 → ok (2.5s)
  [ 16/90] PAC006 rep 1 → ok (2.6s)
  [ 17/90] PAC006 rep 2 → ok (2.7s)
  [ 18/90] PAC006 rep 3 → ok (2.5s)
  [ 19/90] PAC007 rep 1 → ok (2.2s)
  [ 20/90] PAC007 rep 2 → ok (2.2s)
  [ 21/90] PAC007 rep 3 → ok (2.3s)
  [ 22/90] PAC008 rep 1 → ok (2.3s)
  [ 23/90] PAC008 rep 2 → ok (2.2s)
  [ 24/90] PAC008 rep 3 → ok (2.2s)
  [ 25/90] PAC009 rep 1 → ok (2.6s)
  

## 7 · Resultados

- `formato_ok`: fracción de salidas que fueron JSON válido.
- `acuerdo_nivel` (0–1): fracción de repeticiones que coincide con el nivel de alerta más frecuente de ese paciente. 1.0 = el modelo dice siempre lo mismo.
- `latencia_media`: segundos por anotación.

In [9]:
df = pd.read_sql(
    "SELECT * FROM experimento WHERE codigo = ?", con, params=[EXPERIMENTO]
)
print(f"{len(df)} anotaciones del experimento '{EXPERIMENTO}'\n")


def acuerdo_modal(niveles):
    '''Fracción de repeticiones que coincide con el nivel más frecuente.'''
    s = niveles.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


resumen = df.groupby("id_paciente").agg(
    repeticiones=("repeticion", "count"),
    formato_ok=("formato_ok", "mean"),
    acuerdo_nivel=("nivel_alerta", acuerdo_modal),
    latencia_media=("latencia_s", "mean"),
).round(2)

print(f"Formato válido: {df['formato_ok'].mean():.0%}")
print(f"Acuerdo medio del nivel de alerta entre repeticiones: {resumen['acuerdo_nivel'].mean():.2f}")
print(f"Latencia media por anotación: {df['latencia_s'].mean():.1f}s\n")
resumen

90 anotaciones del experimento 'tool_calling-s1-t0.7-20260713'

Formato válido: 99%
Acuerdo medio del nivel de alerta entre repeticiones: 0.93
Latencia media por anotación: 2.3s



,repeticiones,formato_ok,acuerdo_nivel,latencia_media
id_paciente,,,,
PAC001,3,1.00,0.67,2.47
PAC002,3,1.00,1.00,2.37
PAC003,3,1.00,1.00,2.64
PAC004,3,1.00,1.00,2.48
PAC005,3,1.00,1.00,2.65
PAC006,3,1.00,1.00,2.60
PAC007,3,1.00,0.67,2.23
PAC008,3,1.00,1.00,2.22
PAC009,3,1.00,1.00,2.39


In [10]:
# Las anotaciones de un paciente concreto, repetición a repetición
UN_PACIENTE = df["id_paciente"].iloc[0]   # cambiar por el que interese

detalle = df[df["id_paciente"] == UN_PACIENTE]
for _, fila in detalle.iterrows():
    print(f"— repetición {fila.repeticion}: nivel={fila.nivel_alerta} "
          f"items={fila.items_detectados}")
    print(f"  nota: {fila.nota_clinica}\n")

— repetición 0: nivel=alto items=[1, 4, 3]
  nota: Se observa sintomatología significativa en la hiperactividad motora (inquietud constante), déficits notables en la conciencia social del impacto de su conducta y dificultades marcadas en la memoria de trabajo secuencial. Estos hallazgos sugieren un perfil atencional e impulsivo que requiere intervención especializada.

— repetición 1: nivel=alto items=[1, 4, 3]
  nota: Se observa un patrón preocupante de inquietud motora significativa y dificultades notables en la atención secuencial y la conciencia del impacto social. Estos hallazgos sugieren déficits marcados que requieren exploración diagnóstica completa en el contexto TDAH.

— repetición 2: nivel=moderado items=[1, 3, 4]
  nota: Se observa una marcada hiperactividad motora (inquieta/saltando de la silla) y dificultades significativas en la memoria de trabajo al seguir instrucciones secuenciales. La madre reporta también problemas con la conciencia del impacto conductual en sus pare

## 8 · Siguiente paso

Comparar este experimento con los de los otros backends (u otros parámetros) en `05_comparacion_experimentos.ipynb`, usando los códigos de experimento.